<a href="https://colab.research.google.com/github/Varun-gabhane/PredictiveAnalysis-Manufacturing/blob/Varun-gabhane-patch-1/chapter_appendix-tools-for-deep-learning/jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [3]:
from google.colab import files
uploaded = files.upload("/content/Purchase_Order_Quantity_Price_detail_for_Commodity_Goods_procurements_20251008.csv")

Saving Purchase_Order_Quantity_Price_detail_for_Commodity_Goods_procurements_20251008.csv to /content/Purchase_Order_Quantity_Price_detail_for_Commodity_Goods_procurements_20251008.csv/Purchase_Order_Quantity_Price_detail_for_Commodity_Goods_procurements_20251008.csv


In [4]:
import io
filename = list(uploaded.keys())[0]
data = pd.read_csv(io.BytesIO(uploaded[filename]))

In [13]:
print("Dataset loaded!")
print(f"Rows: {data.shape[0]}, Columns: {data.shape[1]}")
print("\n First 5 rows:")
display(data.head())
print("\n Data types:")
print(data.dtypes)

Dataset loaded!
Rows: 307544, Columns: 24

 First 5 rows:


,COMMODITY,COMMODITY_DESCRIPTION,EXTENDED_DESCRIPTION,QUANTITY,UNIT_OF_MEASURE,UNIT_OF_MEAS_DESC,UNIT_PRICE,ITM_TOT_AM,MASTER_AGREEMENT,CONTRACT_NAME,...,AD_LN_1,AD_LN_2,CITY,ST,ZIP,CTRY,DATA_BUILD_DATE,Month,Quarter,Year
0,12025,"Bridges, Passenger Loading",Quote # RQ10351-0,0.00,EA,Each,3157.765537,6999.00,MA8100PS170000001,Onsite Support for GPU8 Troubleshoot and Repair,...,"Oshkosh Aerotech, LLC",4074 South 1900 West,Roy,UT,84067,US,2025-09-29,2,1,2025
1,54523,"Impact Tools, Air Powered (Not Road Building)","2 each Motor, Potable Water Reel #P56SX163 @43...",0.00,EA,Each,3157.765537,872.12,MA7400GC130000008,Motor Potable Water Reel for ABIA MAX PO# 8882...,...,4616 W Howard Ln Ste 965,One Dell Way,AUSTIN,TX,78728-6326,US,2025-09-29,11,4,2017
2,20086,"Uniforms, Cotton","BLOUIN, BENDER, LAZENBY, BUMPUS,WELLER",0.00,EA,Each,3157.765537,671.00,MA7400GA070000017,uniform,...,650 CANION ST,One Dell Way,AUSTIN,TX,78752-3510,US,2025-09-29,6,2,2010
3,28521,"Conduit and Fittings, Aluminum",RC LN_____ QTY DEL_____ P/F_____ B/O______ DEL...,15.00,EA,Each,3.829000,57.44,MA7400GC150000004,"Tape, Fittings/connectors, Channel Strut, wire",...,3206 INDUSTRIAL TERRACE,One Dell Way,AUSTIN,TX,78757-7612,US,2025-09-29,7,3,2013
4,7452114,"ASPHALTIC CONCRETE, HOT MIX, IN ACCORDANCE WIT...",RC LN_____ QTY DEL_____ P/F_____ B/O______ DEL...,81.36,TON,Ton,58.180180,4733.54,MA6200GA080000118,APAC/S & B various section/hot mix,...,1 Chisholm Trail Ste 450,One Dell Way,Round Rock,TX,78681,US,2025-09-29,5,2,2012



 Data types:
COMMODITY                        object
COMMODITY_DESCRIPTION            object
EXTENDED_DESCRIPTION             object
QUANTITY                        float64
UNIT_OF_MEASURE                  object
UNIT_OF_MEAS_DESC                object
UNIT_PRICE                      float64
ITM_TOT_AM                      float64
MASTER_AGREEMENT                 object
CONTRACT_NAME                    object
PURCHASE_ORDER                   object
AWARD_DATE               datetime64[ns]
VENDOR_CODE                      object
LGL_NM                           object
AD_LN_1                          object
AD_LN_2                          object
CITY                             object
ST                               object
ZIP                              object
CTRY                             object
DATA_BUILD_DATE          datetime64[ns]
Month                             int32
Quarter                           int32
Year                              int32
dtype: object


In [7]:
date_cols = ['AWARD_DATE', 'DATA_BUILD_DATE']
for col in date_cols:
    if col in data.columns:
        data[col] = pd.to_datetime(data[col], errors='coerce')

In [8]:
data = data.drop_duplicates()

In [9]:
if 'UNIT_PRICE' in data.columns:
    data['UNIT_PRICE'] = data['UNIT_PRICE'].replace(0, np.nan)

In [10]:
if all(col in data.columns for col in ['ITM_TOT_AM', 'UNIT_PRICE']):
    data['QUANTITY'] = data.apply(
        lambda x: x['ITM_TOT_AM'] / x['UNIT_PRICE'] if pd.isna(x['QUANTITY']) and x['UNIT_PRICE'] > 0 else x['QUANTITY'],
        axis=1
    )

In [11]:
for col in data.columns:
    if data[col].dtype in ['float64', 'int64']:
        data[col] = data[col].fillna(data[col].mean())
    else:
        data[col] = data[col].fillna(data[col].mode()[0])

In [12]:
if 'AWARD_DATE' in data.columns:
    data['Month'] = data['AWARD_DATE'].dt.month
    data['Quarter'] = data['AWARD_DATE'].dt.quarter
    data['Year'] = data['AWARD_DATE'].dt.year

In [14]:
if all(col in data.columns for col in ['ITM_TOT_AM', 'QUANTITY']):
    data['Price_Per_Quantity'] = data['ITM_TOT_AM'] / data['QUANTITY']

In [15]:
if 'Market_Index' in data.columns and 'UNIT_PRICE' in data.columns:
    data['Market_Deviation'] = data['UNIT_PRICE'] - data['Market_Index']

In [16]:
drop_cols = ['EXTENDED_DESCRIPTION', 'AD_LN_1', 'AD_LN_2', 'ZIP', 'DATA_BUILD_DATE']
data = data.drop([c for c in drop_cols if c in data.columns], axis=1, errors='ignore')


In [17]:
print("\n Cleaned Data Sample:")
display(data.head())
print("\n Missing values after cleaning:")
print(data.isnull().sum())


 Cleaned Data Sample:


,COMMODITY,COMMODITY_DESCRIPTION,QUANTITY,UNIT_OF_MEASURE,UNIT_OF_MEAS_DESC,UNIT_PRICE,ITM_TOT_AM,MASTER_AGREEMENT,CONTRACT_NAME,PURCHASE_ORDER,AWARD_DATE,VENDOR_CODE,LGL_NM,CITY,ST,CTRY,Month,Quarter,Year,Price_Per_Quantity
0,12025,"Bridges, Passenger Loading",0.00,EA,Each,3157.765537,6999.00,MA8100PS170000001,Onsite Support for GPU8 Troubleshoot and Repair,DO810025021106325,2025-02-11,V00000978964,"Oshkosh AeroTech, LLC",Roy,UT,US,2,1,2025,inf
1,54523,"Impact Tools, Air Powered (Not Road Building)",0.00,EA,Each,3157.765537,872.12,MA7400GC130000008,Motor Potable Water Reel for ABIA MAX PO# 8882...,DO810017113003484,2017-11-30,BEA0622750,APPLIED INDUSTRIAL TECHNOLOGIES INC,AUSTIN,TX,US,11,4,2017,inf
2,20086,"Uniforms, Cotton",0.00,EA,Each,3157.765537,671.00,MA7400GA070000017,uniform,DO930010061823363,2010-06-18,MIL3235500,MILLER UNIFORMS & EMBLEMS INC,AUSTIN,TX,US,6,2,2010,inf
3,28521,"Conduit and Fittings, Aluminum",15.00,EA,Each,3.829000,57.44,MA7400GC150000004,"Tape, Fittings/connectors, Channel Strut, wire",PO220013070805154,2013-07-09,SUM8300245,SUMMIT ELECTRIC SUPPLY CO INC,AUSTIN,TX,US,7,3,2013,3.829333
4,7452114,"ASPHALTIC CONCRETE, HOT MIX, IN ACCORDANCE WIT...",81.36,TON,Ton,58.180180,4733.54,MA6200GA080000118,APAC/S & B various section/hot mix,DO620012050113149,2012-05-01,VC0000102792,APAC-TEXAS INC,Round Rock,TX,US,5,2,2012,58.180187



 Missing values after cleaning:
COMMODITY                0
COMMODITY_DESCRIPTION    0
QUANTITY                 0
UNIT_OF_MEASURE          0
UNIT_OF_MEAS_DESC        0
UNIT_PRICE               0
ITM_TOT_AM               0
MASTER_AGREEMENT         0
CONTRACT_NAME            0
PURCHASE_ORDER           0
AWARD_DATE               0
VENDOR_CODE              0
LGL_NM                   0
CITY                     0
ST                       0
CTRY                     0
Month                    0
Quarter                  0
Year                     0
Price_Per_Quantity       0
dtype: int64


In [18]:
output_filename = "cleaned_vendor_data.csv"
data.to_csv(output_filename, index=False)
print(f"\n Cleaned dataset saved as '{output_filename}'")


📁 Cleaned dataset saved as 'cleaned_vendor_data.csv'
